# Train-Test-Split
In diesem Notebook wird der Datensatz in Trainings- und Testdaten aufgeteilt. Durch die Aufteilung wird Dataleakage während des Trainings und der Evaluation vermieden.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import shutil

## Daten einlesen

In [2]:
df = pd.read_excel(
    "../data/labels/Libary_Bruchbilder.xlsx", 
    usecols="A, B, D:I"  
)
df['Image_path'] = df['Image No.'].apply(lambda x: x.split('_')[0] + '_FHNW_' + x.split('_')[1] + '.jpg')
df

,Image No.,Rating,Cond.,Product,Pre-treatment 1,Pre-treatment 2,Pre-treatment 3,Substrate,Image_path
0,Image_00001,1,B,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00001.jpg
1,Image_00002,1,C,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00002.jpg
2,Image_00003,1,F,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00003.jpg
3,Image_00004,1,G,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00004.jpg
4,Image_00005,1,L,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00005.jpg
...,...,...,...,...,...,...,...,...,...
1003,Image_01004,1,B,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01004.jpg
1004,Image_01005,1,C,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01005.jpg
1005,Image_01006,2 SF 15%,F,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01006.jpg
1006,Image_01007,1,G,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01007.jpg


## Daten säubern
Da die Daten eine grosse Class-Inbalance haben, wird ein Stratified-Split durchgeführt. Dazu werden die Daten mit der gleichen Verteilung der Labels aufgeteilt. Deshalb muss die Label-Spalte gesäubert werden.

Beim säubern gehen 33 Datenpunkte verlohren, da diese kein numerisches Rating enthalten haben. Diese Datenpunkte werden momentan nicht weiter berücksichtigt.
- Update 15.04.2025: Die Datenpunkte werden nach Absprache mit dem Kunden nach einer Regel klassifiziert. 

Ergänzung der leeren Ratings. Details: siehe ./notebooks/00_eda_labels.ipynb

In [3]:
df_clean = df.copy()
df_clean['Rating_clean'] = df.apply(
    lambda row: 5 if not(str(row['Rating'])[0].isdigit()) and row['Rating'] == 'SCF'
    else (1 if not(str(row['Rating'])[0].isdigit()) else str(row['Rating'])[0]),
    axis=1
)
df_clean['Adhesive'] = df_clean['Rating'].apply(lambda x: f'{x}'.split(' ')[1:])
df_clean

,Image No.,Rating,Cond.,Product,Pre-treatment 1,Pre-treatment 2,Pre-treatment 3,Substrate,Image_path,Rating_clean,Adhesive
0,Image_00001,1,B,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00001.jpg,1,[]
1,Image_00002,1,C,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00002.jpg,1,[]
2,Image_00003,1,F,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00003.jpg,1,[]
3,Image_00004,1,G,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00004.jpg,1,[]
4,Image_00005,1,L,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00005.jpg,1,[]
...,...,...,...,...,...,...,...,...,...,...,...
1003,Image_01004,1,B,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01004.jpg,1,[]
1004,Image_01005,1,C,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01005.jpg,1,[]
1005,Image_01006,2 SF 15%,F,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01006.jpg,2,"[SF, 15%]"
1006,Image_01007,1,G,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01007.jpg,1,[]


In [4]:
import ast
import re

def extract_first(s):
    if len(s) == 0:
        return 'Empty'
    first = s[0]

    cleaned = re.sub(r'[^A-Za-z]', '', first)

    return cleaned or 'Empty'

df_clean['additional_rating'] = df_clean['Adhesive'].apply(extract_first)

In [5]:
df_clean

,Image No.,Rating,Cond.,Product,Pre-treatment 1,Pre-treatment 2,Pre-treatment 3,Substrate,Image_path,Rating_clean,Adhesive,additional_rating
0,Image_00001,1,B,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00001.jpg,1,[],Empty
1,Image_00002,1,C,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00002.jpg,1,[],Empty
2,Image_00003,1,F,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00003.jpg,1,[],Empty
3,Image_00004,1,G,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00004.jpg,1,[],Empty
4,Image_00005,1,L,Sikaflex-668,SCP,SA-100,SP-507,Glas,Image_FHNW_00005.jpg,1,[],Empty
...,...,...,...,...,...,...,...,...,...,...,...,...
1003,Image_01004,1,B,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01004.jpg,1,[],Empty
1004,Image_01005,1,C,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01005.jpg,1,[],Empty
1005,Image_01006,2 SF 15%,F,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01006.jpg,2,"[SF, 15%]",SF
1006,Image_01007,1,G,Sikaflex-554 PC,NaN,SA-205,NaN,GFK,Image_FHNW_01007.jpg,1,[],Empty


In [6]:
df_clean['Rating_clean'] = pd.to_numeric(df_clean['Rating_clean'], errors='coerce')

In [7]:
df_clean.groupby('Rating_clean').size().reset_index(name='Count')

,Rating_clean,Count
0,1,732
1,2,67
2,3,66
3,4,24
4,5,119


In [8]:
df_clean['additional_rating'].value_counts()

additional_rating
Empty    919
SCF       27
ES        20
PaIF      18
SCPrF     13
TU         4
SF         4
CFPa       2
PaDF       1
Name: count, dtype: int64

## Split der Daten
Es werden 20% der Daten für das Testset verwendet. Das Stratify wird auf der Rating_clean-Spalte durchgeführt.

In [9]:
train_val_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    stratify=df_clean['Rating_clean'],
    random_state=42 
)

In [10]:
train_val_df

,Image No.,Rating,Cond.,Product,Pre-treatment 1,Pre-treatment 2,Pre-treatment 3,Substrate,Image_path,Rating_clean,Adhesive,additional_rating
529,Image_00530,5,F,Sikaflex-668 PC,SCP,Plasma 2,NaN,PP,Image_FHNW_00530.jpg,5,[],Empty
620,Image_00621,1,G,Sikaflex-668 PC,SCP,NaN,SP-507,Pulverlack,Image_FHNW_00621.jpg,1,[],Empty
464,Image_00465,1,F,Sikaflex-252,AP-C,NaN,SP-207,Alu roh,Image_FHNW_00465.jpg,1,[],Empty
768,Image_00769,1,L,Sikaflex-268,SCP,NaN,SP-207,Alu,Image_FHNW_00769.jpg,1,[],Empty
560,Image_00561,1,F,Sikaflex-268,SCP,SA-100,NaN,Pulverlack,Image_FHNW_00561.jpg,1,[],Empty
...,...,...,...,...,...,...,...,...,...,...,...,...
370,Image_00371,1,L,Sikaflex-223,Benzin,NaN,SP-207,Pulverlack,Image_FHNW_00371.jpg,1,[],Empty
306,Image_00307,1,B,Sikaflex-554 PC,SCP,NaN,SP-207,Alu,Image_FHNW_00307.jpg,1,[],Empty
809,Image_00810,1,G,Sikaflex-552 AT,AP-C,SA-205,NaN,HPL,Image_FHNW_00810.jpg,1,[],Empty
344,Image_00345,1,G,SikaTack-Go!,Benzin,NaN,SP-207,Pulverlack,Image_FHNW_00345.jpg,1,[],Empty


In [11]:
test_df

,Image No.,Rating,Cond.,Product,Pre-treatment 1,Pre-treatment 2,Pre-treatment 3,Substrate,Image_path,Rating_clean,Adhesive,additional_rating
89,Image_00090,1,G,Sikaflex-228,NaN,NaN,NaN,Alu,Image_FHNW_00090.jpg,1,[],Empty
962,Image_00963,1,C,Sikaflex-521 UV,NaN,SA-205,NaN,GFK,Image_FHNW_00963.jpg,1,[],Empty
792,Image_00793,1,B,Sikaflex-552 AT,AP-C,SA-205,NaN,Edelstahl,Image_FHNW_00793.jpg,1,[],Empty
372,Image_00373,1,C,SikaTack-Go!,Benzin,SA-306 LUM,NaN,Pulverlack,Image_FHNW_00373.jpg,1,[],Empty
319,Image_00320,1,G,Sikaflex-223,Benzin,SA-306 LUM,NaN,Pulverlack,Image_FHNW_00320.jpg,1,[],Empty
...,...,...,...,...,...,...,...,...,...,...,...,...
857,Image_00858,1,G,Sikaflex-668 PC,SCP,SA-100,SP-507,GFK,Image_FHNW_00858.jpg,1,[],Empty
40,Image_00041,1,C,Sikaflex-668 PC,SCP,NaN,SP-507,Glas,Image_FHNW_00041.jpg,1,[],Empty
643,Image_00644,1,B,Sikaflex-668,SCP,NaN,SP-507,Pulverlack,Image_FHNW_00644.jpg,1,[],Empty
66,Image_00067,1,B,Sikaflex-668,SCP,SA-100,SP-206GP,Glas,Image_FHNW_00067.jpg,1,[],Empty


## Verteilung der Labels

In [12]:
print(test_df['Rating_clean'].value_counts(normalize=True))
print(df_clean['Rating_clean'].value_counts(normalize=True))

Rating_clean
1    0.727723
5    0.118812
2    0.064356
3    0.064356
4    0.024752
Name: proportion, dtype: float64
Rating_clean
1    0.726190
5    0.118056
2    0.066468
3    0.065476
4    0.023810
Name: proportion, dtype: float64


## Kopieren der Bilder in separaten Ordner
Die Bilder werden in die train- und test-Ordner kopiert.

In [13]:
new_dir = '../data/test/img/'
original_dir = '../data/img/'
for _, row in test_df.iterrows():
    src = original_dir + row['Image_path']
    shutil.copy(src, new_dir)
    
new_dir = '../data/train/img/'
for _, row in train_val_df.iterrows():
    src = original_dir + row['Image_path']
    shutil.copy(src, new_dir)
    
        

Ebenfalls werden die Labels in Excel-Dateien gespeichert und in die gleiche Ordnerstruktur kopiert.

In [14]:
test_df.to_excel("../data/test/labels/test.xlsx", index=False)
train_val_df.to_excel("../data/train/labels/train_val.xlsx", index=False)